# 확장 연구: 생성 표본의 선명도 개선

본편(`imagegeneration.ipynb`)의 Chapter 3 §3.7.3 은 표본이 뭉개져 보이는 원인을 둘로 나눴다.

1. **학습 예산 부족** — 파라미터 14배, 최적화 스텝 68배 부족. 총 연산량으로 약 3자릿수 차이
2. **평균으로의 회귀** — MSE 목적함수 아래에서 불확실한 영역의 최적 예측은 가능한 결과들의 평균이고, 평균은 매끄럽다

이 노트북은 **확산 프레임워크와 신경망 구조를 그대로 두고 학습·샘플링 하이퍼파라미터만 조정**해
두 원인을 어디까지 완화할 수 있는지 측정한다. 여섯 단계의 누적 사다리로 구성해
각 변경이 얼마나 기여했는지 분리할 수 있게 했다.

## 기대치에 대한 정직한 말

이 확장으로 논문 수준(FID 3 내외)에 도달하지는 않는다. 현실적인 목표는
**CIFAR-10 FID 94.9 → 30~45 구간**이며, 이 정도면 표본이 눈에 띄게 선명해지지만
사진 같지는 않다. 3자릿수 연산량 격차 전부를 소비자용 GPU 몇 시간으로 메울 수는 없다.

## 실행 전 확인

* 본편의 체크포인트(`model/CIFAR-10/`, `model/CelebA/`)는 **건드리지 않는다**.
  이 노트북은 `model/extension/` 에만 쓴다. Chapter 3 의 근거 자료는 그대로 보존된다.
* 데이터 캐시(`data/cache/*.pt`)는 본편이 만들어 둔 것을 재사용한다. CelebA 재디코딩이 없다.
* 중단되어도 **에폭 단위로 이어서 학습**한다. 같은 셀을 다시 실행하면 남은 에폭만 돈다.
* 동시 실행 방지 잠금이 걸려 있다. 두 곳에서 동시에 돌리면 체크포인트가 섞인다.

In [ ]:
import json
import math
import os
import time
import unicodedata
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from torchvision.utils import make_grid

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---- 가속 설정 (본편과 동일. 정확도에 영향 없음) -------------------------
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
USE_AMP = DEVICE.type == "cuda" and torch.cuda.is_bf16_supported()
AMP_DTYPE = torch.bfloat16
# torch.compile 은 Triton 이 필요하다. 이 환경에서는 없어서 실패한다.
#   pip install triton-windows
# 를 설치했다면 True 로 바꿔 볼 것. 실패해도 학습은 그대로 진행된다.
USE_COMPILE = False

# ---- 경로. 본편 산출물과 완전히 분리한다 --------------------------------
CACHE_DIR = Path("data/cache")            # 본편이 만든 캐시를 읽기만 한다
EXT_DIR = Path("model/extension")         # 이 노트북이 쓰는 유일한 모델 경로
FIG_DIR = Path("dissertation/imgs")       # 논문에 넣을 그림
RESULT_JSON = EXT_DIR / "extension_study.json"
for d in (EXT_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

# ---- 고정 조건 (본편과 같아야 비교가 성립한다) --------------------------
NUM_STEPS = 1000        # 확산 스텝 수 T
BATCH_SIZE = 128
BASE_LR = 2e-4
TIME_DIM = 128
SEED = 0                # 본편의 시드 0 결과와 직접 비교한다

# ---- 평가 조건 -----------------------------------------------------------
FID_SAMPLES = 2048      # 본편과 같은 값. 바꾸면 본편 수치와 비교할 수 없다
# "ancestral" 은 본편과 같은 1000 스텝 샘플러라 94.93 과 직접 비교된다.
# "ddim" 으로 바꾸면 샘플링이 약 10배 빨라지지만(전체의 18% -> 2%) 본편 수치와
# 비교할 수 없게 된다. 실측 2048장 생성 시간은 아래 예산 표 참조.
EVAL_SAMPLER = "ancestral"
EVAL_DDIM_STEPS = 100

# ---- 데이터셋별 기본 폭 (본편과 동일) ------------------------------------
DATASETS = {
    "CIFAR-10": dict(image_size=32, channels=(64, 128), limit=None),
    "CelebA":   dict(image_size=64, channels=(32, 64, 128), limit=50_000),
}

# ---- 본편 시드 0 결과를 기준선으로 읽어 둔다 -----------------------------
BASELINE = {}
_bp = Path("model/seed_study.json")
if _bp.exists():
    _raw = json.loads(_bp.read_text(encoding="utf-8"))
    for _k, _v in _raw.items():
        _ds, _arch, _seed = _k.split("|")
        if _arch == "resunet" and _seed == "0":
            BASELINE[_ds] = _v
    print("기준선(본편 시드 0, ResUNet):")
    for _ds, _v in BASELINE.items():
        print(f"  {_ds:9s} FID {_v['fid']:7.2f} | IS {_v['is_mean']:.3f} | "
              f"loss {_v['loss_final']:.5f}")
else:
    print("경고: model/seed_study.json 이 없어 기준선 수치를 읽지 못했다")

print()
print("device:", DEVICE, "| AMP:", USE_AMP)
print("모델 저장 경로:", EXT_DIR, "(본편 체크포인트와 분리됨)")

## 1. 무엇을 바꾸는가

여섯 가지를 바꾼다. 각각이 위의 두 원인 중 어디를 겨냥하는지 함께 적는다.

| 변경 | 값 | 겨냥하는 원인 | 근거 |
|---|---|---|---|
| **가중치 EMA** | decay 0.9995, 워밍업 보정 | 평균으로의 회귀 | 마지막 반복의 흔들림을 없앤다. 확산 모델 구현에서 사실상 표준이며 FID 를 크게 낮춘다 |
| **학습률 워밍업** | 선형 2000 스텝 | 예산 | 초기 발산을 막아 같은 스텝에서 더 낮은 손실에 도달한다 (Ho et al. 는 5000 스텝) |
| **기울기 클리핑** | L2 노름 1.0 | 예산 | 고노이즈 구간의 큰 기울기가 학습을 되돌리는 것을 막는다 |
| **드롭아웃** | 0.1 | 평균으로의 회귀 | Ho et al. 가 CIFAR-10 에 쓴 값. 긴 학습에서 과적합을 억제한다 |
| **min-SNR 가중** | γ = 5 | 예산 | 타임스텝별 기여를 재조정한다. Hang et al. 는 고정 FID 도달까지 수 배 빨라진다고 보고한다 |
| **용량과 학습량** | 폭 ↑, 단계 ↑, 에폭 30 → 150 | 예산 | 원인 1 을 정면으로 다룬다. 세 원인 중 가장 확실하지만 가장 비싸다 |

### min-SNR 가중에 대한 주의

ε 예측에서 가중치는 `min(SNR_t, γ) / SNR_t` 이며, SNR_t = ᾱ_t / (1 − ᾱ_t) 다.
γ → ∞ 이면 가중치가 1 이 되어 표준 손실과 정확히 같아진다. γ = 5 는 ᾱ > 5/6,
즉 코사인 스케줄에서 노이즈가 가장 적은 앞쪽 약 20% 의 타임스텝을 낮게 친다.

이 기법의 효과는 **가정하지 않고 측정한다**. 사다리에서 자기 단계를 차지하므로
도움이 되지 않으면 결과 표에 그대로 드러난다.

### 샘플러

FID 는 본편과 같은 조상 샘플링(1000 스텝)으로 계산한다. 샘플러를 바꾸면 FID 가
바뀌므로, 그래야 본편 수치와 비교가 성립한다. DDIM 은 최종 설정에 대해서만
**별도 행**으로 보고해 속도와 품질의 교환을 보인다.

In [ ]:
def raw(model):
    """torch.compile 로 감싼 모델에서 원본을 꺼낸다. 체크포인트 호환을 위해서다."""
    return getattr(model, "_orig_mod", model)


class EMA:
    """가중치의 지수이동평균.

    짧은 학습에서 decay 를 그대로 쓰면 평균이 초기값에 붙들린다. 스텝 수에 따라
    decay 를 서서히 올려(워밍업) 초반에도 의미 있는 평균이 되도록 한다.
    decay 0.9995 의 시정수는 약 2000 스텝으로, 이 노트북의 학습 길이에 맞다.
    (Ho et al. 의 0.9999 는 80만 스텝 학습을 전제한 값이라 여기서는 너무 느리다)

    구현 주의: 텐서마다 mul_/add_ 를 부르면 스텝마다 커널이 수십 번 뜨고
    state_dict() 도 매번 새 OrderedDict 를 만든다. 실측으로 스텝의 5.9%(2.5M),
    3.0%(31.5M)를 잡아먹었다. 텐서 목록을 한 번만 만들고 _foreach 로 묶으면
    오버헤드가 사실상 사라진다. 옵티마이저가 파라미터를 제자리 갱신하므로
    텐서 객체는 학습 내내 그대로여서 목록을 재사용해도 안전하다.
    """

    def __init__(self, model, decay=0.9995):
        self.decay = decay
        self.step = 0
        sd = raw(model).state_dict()
        self.shadow = {k: v.detach().clone().float() for k, v in sd.items()}
        self._bind(sd)

    def _bind(self, sd):
        """실가중치와 그림자 가중치를 _foreach 가 쓸 목록으로 묶는다."""
        fk = [k for k, v in sd.items() if v.dtype.is_floating_point]
        ok = [k for k, v in sd.items() if not v.dtype.is_floating_point]
        self._live = [sd[k] for k in fk]
        self._sh = [self.shadow[k] for k in fk]          # 같은 객체를 참조한다
        self._olive = [sd[k] for k in ok]
        self._osh = [self.shadow[k] for k in ok]

    @torch.no_grad()
    def update(self, model=None):
        self.step += 1
        d = min(self.decay, (1.0 + self.step) / (10.0 + self.step))
        torch._foreach_mul_(self._sh, d)
        torch._foreach_add_(self._sh, self._live, alpha=1.0 - d)
        for dst, src in zip(self._osh, self._olive):     # 정수 버퍼는 그대로 복사
            dst.copy_(src)

    def copy_to(self, model):
        """평균 가중치를 모델에 싣는다. 샘플링 직전에 호출한다."""
        sd = raw(model).state_dict()
        raw(model).load_state_dict({k: self.shadow[k].to(sd[k].dtype) for k in sd})

    def state_dict(self):
        return {"decay": self.decay, "step": self.step, "shadow": self.shadow}

    def load_state_dict(self, sd, model=None):
        self.decay, self.step, self.shadow = sd["decay"], sd["step"], sd["shadow"]
        if model is not None:                 # 이어서 학습할 때 목록을 다시 묶는다
            self._bind(raw(model).state_dict())


def lr_at(step, base_lr, warmup):
    """선형 워밍업. warmup=0 이면 항상 base_lr 를 준다."""
    if warmup <= 0:
        return base_lr
    return base_lr * min(1.0, (step + 1) / warmup)


def min_snr_weight(alphas_bar, t, gamma):
    """ε 예측용 min-SNR 가중치 min(SNR,γ)/SNR. gamma=None 이면 1 을 준다."""
    if gamma is None:
        return None
    ab = alphas_bar.gather(0, t)
    snr = ab / (1.0 - ab)
    return (snr.clamp(max=gamma) / snr).view(-1, 1, 1, 1)


def pad(text, width):
    """표 정렬용. 한글은 한 글자가 두 칸이라 f-string 의 :<n 이 어긋난다."""
    w = sum(2 if unicodedata.east_asian_width(ch) in "WF" else 1 for ch in text)
    return text + " " * max(0, width - w)

## 2. 신경망

본편의 `ResUNet` 과 같은 구조다. 하이퍼파라미터로 노출한 것만 다르다.

* `dropout` — residual block 의 마지막 컨볼루션 직전. Ho et al. 의 배치와 같다
* `blocks_per_stage` — 해상도 단계마다 residual block 을 몇 개 쌓을지
* `channels` — 단계별 폭. 길이가 곧 해상도 단계 수다

`residual=False` 를 주면 본편의 비교군 A(plain UNet)가 그대로 나온다.
파라미터 수와 연산량은 residual 여부와 무관하다.

**드롭아웃을 켰으므로 샘플링과 평가 전에 반드시 `model.eval()` 을 불러야 한다.**
본편에는 드롭아웃이 없어 문제가 되지 않았지만 여기서는 결과가 달라진다.
아래 샘플링·평가 함수는 모두 `eval()` 을 명시적으로 호출한다.

In [ ]:
def timestep_embedding(t, dim):
    """정현파 timestep 임베딩. 본편과 동일."""
    half = dim // 2
    freqs = torch.exp(-math.log(10000.0) * torch.arange(half, device=t.device) / half)
    args = t.float()[:, None] * freqs[None, :]
    return torch.cat([torch.cos(args), torch.sin(args)], dim=-1)


def gn(channels, max_groups=8):
    for g in (max_groups, 4, 2, 1):
        if channels % g == 0:
            return nn.GroupNorm(g, channels)


class ConvBlock(nn.Module):
    """2층 컨볼루션 블록. residual=False 면 shortcut 만 빠진 plain 블록이 된다."""

    def __init__(self, channels, time_dim, residual=True, dropout=0.0):
        super().__init__()
        self.residual = residual
        self.norm1 = gn(channels)
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1)
        self.time_proj = nn.Linear(time_dim, channels)
        self.norm2 = gn(channels)
        # 드롭아웃은 마지막 컨볼루션 직전에 둔다 (Ho et al. 의 배치)
        self.drop = nn.Dropout(dropout) if dropout > 0 else nn.Identity()
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1)

    def forward(self, x, t_emb):
        h = self.conv1(F.silu(self.norm1(x)))
        h = h + self.time_proj(F.silu(t_emb))[:, :, None, None]
        h = self.conv2(self.drop(F.silu(self.norm2(h))))
        return x + h if self.residual else h


class Stage(nn.Module):
    """블록을 n 개 쌓은 하나의 해상도 단계."""

    def __init__(self, channels, time_dim, n, residual, dropout):
        super().__init__()
        self.blocks = nn.ModuleList(
            [ConvBlock(channels, time_dim, residual, dropout) for _ in range(n)])

    def forward(self, x, t_emb):
        for b in self.blocks:
            x = b(x, t_emb)
        return x


class Encoder(nn.Module):
    def __init__(self, in_ch, out_ch, time_dim, n, residual, dropout):
        super().__init__()
        self.conv = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.stage = Stage(out_ch, time_dim, n, residual, dropout)

    def forward(self, x, t_emb):
        return self.stage(self.conv(x), t_emb)


class Decoder(nn.Module):
    """업샘플 후 대응 encoder 특징을 concat 한다."""

    def __init__(self, in_ch, skip_ch, out_ch, time_dim, n, residual, dropout):
        super().__init__()
        self.upconv = nn.ConvTranspose2d(in_ch, out_ch, 2, stride=2)
        self.stage = Stage(out_ch + skip_ch, time_dim, n, residual, dropout)

    def forward(self, x, skip, t_emb):
        x = self.upconv(x)
        if x.shape[2:] != skip.shape[2:]:
            x = F.interpolate(x, size=skip.shape[2:], mode="bilinear", align_corners=False)
        return self.stage(torch.cat([x, skip], dim=1), t_emb)


class ResUNet(nn.Module):
    def __init__(self, channels=(64, 128), time_dim=TIME_DIM, residual=True,
                 dropout=0.0, blocks_per_stage=1):
        super().__init__()
        self.time_dim = time_dim
        self.time_mlp = nn.Sequential(
            nn.Linear(time_dim, time_dim), nn.SiLU(), nn.Linear(time_dim, time_dim))

        self.encoders = nn.ModuleList()
        in_ch = 3
        for c in channels:
            self.encoders.append(
                Encoder(in_ch, c, time_dim, blocks_per_stage, residual, dropout))
            in_ch = c
        self.pool = nn.MaxPool2d(2)
        self.bridge = Stage(channels[-1], time_dim, blocks_per_stage, residual, dropout)

        self.decoders = nn.ModuleList()
        ch = channels[-1]
        for c in reversed(channels):
            self.decoders.append(
                Decoder(ch, c, c, time_dim, blocks_per_stage, residual, dropout))
            ch = c + c
        self.final_conv = nn.Conv2d(ch, 3, 1)

    def forward(self, x, t):
        t_emb = self.time_mlp(timestep_embedding(t, self.time_dim))
        skips = []
        for enc in self.encoders:
            x = enc(x, t_emb)
            skips.append(x)
            x = self.pool(x)
        x = self.bridge(x, t_emb)
        for dec in self.decoders:
            x = dec(x, skips.pop(), t_emb)
        return self.final_conv(x)


def count_params(m):
    return sum(p.numel() for p in m.parameters())

## 3. 확산 과정

본편과 같은 코사인 스케줄, 같은 조상 샘플러다. 두 가지가 추가된다.

1. **min-SNR 가중을 받는 손실** — `p_losses(x0, snr_gamma)`
2. **DDIM 샘플러** — 결정론적(η=0) 역과정. 1000 스텝 대신 50~250 스텝으로 표본을 얻는다

역과정의 `x0` 클리핑은 본편에서 밝힌 대로 유지한다. 이것이 없으면
`1/√α_t` 가 t=999 에서 31.6배라 역과정이 발산한다.

In [ ]:
def cosine_beta_schedule(num_steps, s=0.008):
    """Nichol & Dhariwal 코사인 스케줄. 본편과 동일."""
    steps = torch.arange(num_steps + 1, dtype=torch.float64) / num_steps
    f = torch.cos((steps + s) / (1 + s) * math.pi / 2) ** 2
    alphas_bar = f / f[0]
    betas = 1 - alphas_bar[1:] / alphas_bar[:-1]
    return betas.clamp(max=0.999).float()


class GaussianDiffusion(nn.Module):
    def __init__(self, model, num_steps=NUM_STEPS):
        super().__init__()
        self.model = model
        self.num_steps = num_steps

        betas = cosine_beta_schedule(num_steps)
        alphas = 1.0 - betas
        alphas_bar = torch.cumprod(alphas, dim=0)
        alphas_bar_prev = F.pad(alphas_bar[:-1], (1, 0), value=1.0)

        self.register_buffer("betas", betas)
        self.register_buffer("alphas", alphas)
        self.register_buffer("alphas_bar", alphas_bar)
        self.register_buffer("alphas_bar_prev", alphas_bar_prev)
        self.register_buffer("sqrt_ab", alphas_bar.sqrt())
        self.register_buffer("sqrt_1mab", (1.0 - alphas_bar).sqrt())
        self.register_buffer("posterior_var",
                             betas * (1.0 - alphas_bar_prev) / (1.0 - alphas_bar))
        self.register_buffer("post_c0", alphas_bar_prev.sqrt() * betas / (1.0 - alphas_bar))
        self.register_buffer("post_ct",
                             alphas.sqrt() * (1.0 - alphas_bar_prev) / (1.0 - alphas_bar))

    @staticmethod
    def _gather(buf, t):
        return buf.gather(0, t).view(-1, 1, 1, 1)

    def q_sample(self, x0, t, noise):
        return self._gather(self.sqrt_ab, t) * x0 + self._gather(self.sqrt_1mab, t) * noise

    def p_losses(self, x0, snr_gamma=None):
        """DDPM simple loss. snr_gamma 가 주어지면 타임스텝별로 가중한다."""
        b = x0.size(0)
        t = torch.randint(0, self.num_steps, (b,), device=x0.device)
        noise = torch.randn_like(x0)
        x_t = self.q_sample(x0, t, noise)
        pred = self.model(x_t, t)

        if snr_gamma is None:
            return F.mse_loss(pred, noise)
        # 가중은 float 로 계산한다. bf16 에서 SNR 이 큰 앞쪽 t 는 정밀도가 모자란다
        w = min_snr_weight(self.alphas_bar, t, snr_gamma)      # float32 로 유지한다
        return (w * (pred.float() - noise.float()) ** 2).mean()

    def predict_x0(self, x_t, t, eps):
        return (x_t - self._gather(self.sqrt_1mab, t) * eps) / self._gather(self.sqrt_ab, t)

    def _eps(self, x, t):
        with torch.autocast(device_type=x.device.type, dtype=AMP_DTYPE, enabled=USE_AMP):
            return self.model(x, t).float()

    @torch.no_grad()
    def p_sample_loop(self, shape, device=None, keep_every=None, generator=None):
        """조상 샘플링. 본편과 동일한 절차이며 FID 비교의 기준이 된다."""
        self.model.eval()                      # 드롭아웃을 끈다
        device = device or self.betas.device
        x = torch.randn(shape, device=device, generator=generator)
        traj = [x.cpu()] if keep_every else None

        for step in reversed(range(self.num_steps)):
            t = torch.full((shape[0],), step, dtype=torch.long, device=device)
            eps = self._eps(x, t)
            # x0 를 데이터 범위로 자른다. 없으면 역과정이 발산한다 (본편 §3.7.1)
            x0 = self.predict_x0(x, t, eps).clamp(-1.0, 1.0)
            mean = self._gather(self.post_c0, t) * x0 + self._gather(self.post_ct, t) * x
            if step > 0:
                var = self._gather(self.posterior_var, t)
                x = mean + var.sqrt() * torch.randn(x.shape, device=device,
                                                    generator=generator)
            else:
                x = mean
            if keep_every and (step % keep_every == 0 or step == 0):
                traj.append(x.cpu())

        return (x, traj) if keep_every else x

    @torch.no_grad()
    def ddim_sample(self, shape, steps=100, eta=0.0, device=None, generator=None):
        """DDIM 역과정. eta=0 이면 결정론적이며 스텝 수를 크게 줄일 수 있다.

        ᾱ 를 균등 간격으로 부분추출한다. 마지막 구간의 ᾱ_prev 는 1 로 두어
        (t = -1 에 해당) 마지막 스텝이 곧 x0 가 되게 한다.
        """
        self.model.eval()
        device = device or self.betas.device
        seq = torch.linspace(0, self.num_steps - 1, steps).round().long().flip(0)
        x = torch.randn(shape, device=device, generator=generator)

        for i, cur in enumerate(seq):
            t = torch.full((shape[0],), int(cur), dtype=torch.long, device=device)
            eps = self._eps(x, t)
            ab_t = self.alphas_bar[cur]
            ab_prev = self.alphas_bar[seq[i + 1]] if i + 1 < len(seq) \
                else torch.tensor(1.0, device=device)

            x0 = ((x - (1 - ab_t).sqrt() * eps) / ab_t.sqrt()).clamp(-1.0, 1.0)
            sigma = eta * ((1 - ab_prev) / (1 - ab_t)).sqrt() * (1 - ab_t / ab_prev).sqrt() \
                if (eta > 0 and i + 1 < len(seq)) else torch.tensor(0.0, device=device)
            dir_xt = (1 - ab_prev - sigma ** 2).clamp(min=0).sqrt() * eps
            x = ab_prev.sqrt() * x0 + dir_xt
            if eta > 0 and i + 1 < len(seq):
                x = x + sigma * torch.randn(x.shape, device=device, generator=generator)

        return x

## 4. 자체 점검

학습을 시작하기 전에 새로 넣은 요소가 수학적으로 맞는지 확인한다.
전부 작은 텐서로 몇 초 안에 끝나며, CPU 에서도 돈다.

가장 강한 검사는 **완전한 모델을 가정한 왕복 검사**다. 진짜 노이즈를 그대로
돌려주는 가짜 모델을 끼우면, 조상 샘플러와 DDIM 샘플러 모두 원본 `x0` 를
정확히 되돌려야 한다. 되돌리지 못하면 샘플러 유도식이 틀린 것이다.

In [ ]:
def self_check():
    torch.manual_seed(0)
    dev = torch.device("cpu")          # 점검은 CPU 에서 결정론적으로 한다

    # ---- 1. 신경망 입출력 모양과 파라미터 동등성 ------------------------
    net = ResUNet(channels=(8, 16), dropout=0.1, blocks_per_stage=2)
    plain = ResUNet(channels=(8, 16), dropout=0.1, blocks_per_stage=2, residual=False)
    x = torch.randn(2, 3, 16, 16)
    t = torch.randint(0, NUM_STEPS, (2,))
    assert net(x, t).shape == x.shape, "출력 모양 불일치"
    assert count_params(net) == count_params(plain), \
        "항등 shortcut 은 파라미터를 만들지 않아야 한다"

    # ---- 2. 드롭아웃이 eval 에서 꺼지는가 -------------------------------
    net.eval()
    with torch.no_grad():
        a, b = net(x, t), net(x, t)
    assert torch.equal(a, b), "eval 모드에서 출력이 달라진다 - 드롭아웃이 켜져 있다"
    net.train()
    with torch.no_grad():
        assert not torch.equal(net(x, t), net(x, t)), "train 모드에서 드롭아웃이 동작하지 않는다"

    # ---- 3. 전방 과정 통계 ----------------------------------------------
    diff = GaussianDiffusion(net, num_steps=NUM_STEPS).to(dev)
    x0 = torch.randn(512, 3, 8, 8)
    tT = torch.full((512,), NUM_STEPS - 1, dtype=torch.long)
    xT = diff.q_sample(x0, tT, torch.randn_like(x0))
    assert abs(xT.std().item() - 1.0) < 0.05, f"x_T 표준편차 {xT.std():.3f} (1 이어야 함)"
    assert abs(xT.mean().item()) < 0.05, "x_T 평균이 0 이 아니다"

    # ---- 4. predict_x0 의 역산 정확도 ------------------------------------
    # predict_x0 는 1/sqrt(ab_t) 를 곱하므로 float32 오차가 그만큼 증폭된다.
    # 코사인 스케줄에서 ab_999 = 2.4e-9 이라 증폭률이 2만 배다. 알고리즘 문제가
    # 아니라 조건수 문제이므로(fp64 에서는 1e-11) 허용 오차를 증폭률에 비례시킨다.
    eps = torch.randn_like(x0)
    for step in (10, 500, 900, 999):
        ts = torch.full((512,), step, dtype=torch.long)
        rec = diff.predict_x0(diff.q_sample(x0, ts, eps), ts, eps)
        tol = 20 * 1.2e-7 / diff.alphas_bar[step].sqrt().item()
        err = (rec - x0).abs().max().item()
        assert err < tol, f"t={step} 에서 x0 역산 오차 {err:.2e} (허용 {tol:.2e})"
    # 역과정에서 x0 를 자르는 이유가 하나 더 있다. 모델 오차뿐 아니라 이 조건수도
    # t 가 클수록 커지므로, 자르지 않으면 두 오차가 함께 증폭된다
    assert diff.alphas_bar[-1] < 1e-6, "스케줄 끝에서 ab 가 충분히 작지 않다"

    # ---- 5. 완전한 모델을 가정한 샘플러 왕복 검사 ------------------------
    # 진짜 노이즈를 돌려주는 오라클을 끼운다. 두 샘플러 모두 x0 를 복원해야 한다
    truth = torch.randn(4, 3, 8, 8)
    fixed_eps = torch.randn_like(truth)

    class Oracle(nn.Module):
        def forward(self, x, t):
            return fixed_eps

    oracle = GaussianDiffusion(Oracle(), num_steps=NUM_STEPS).to(dev)
    start = oracle.q_sample(truth.clamp(-1, 1), tT[:4], fixed_eps)

    got = oracle.ddim_sample(truth.shape, steps=50, eta=0.0)   # 자체 노이즈로 시작
    assert torch.isfinite(got).all(), "DDIM 이 발산했다"
    # 시작점을 지정한 왕복: x_T 에서 출발해 x0 로 돌아오는지 직접 확인한다
    xr = start.clone()
    seq = torch.linspace(0, NUM_STEPS - 1, 50).round().long().flip(0)
    for i, cur in enumerate(seq):
        tc = torch.full((4,), int(cur), dtype=torch.long)
        ab_t = oracle.alphas_bar[cur]
        ab_p = oracle.alphas_bar[seq[i + 1]] if i + 1 < len(seq) else torch.tensor(1.0)
        x0h = ((xr - (1 - ab_t).sqrt() * fixed_eps) / ab_t.sqrt()).clamp(-1.0, 1.0)
        xr = ab_p.sqrt() * x0h + (1 - ab_p).clamp(min=0).sqrt() * fixed_eps
    err = (xr - truth.clamp(-1, 1)).abs().max().item()
    assert err < 1e-2, f"완전한 모델에서도 DDIM 왕복 오차가 {err:.4f}"

    # ---- 6. min-SNR 가중 -------------------------------------------------
    tt = torch.arange(0, NUM_STEPS, 100)
    w_inf = min_snr_weight(diff.alphas_bar, tt, 1e9).flatten()
    assert torch.allclose(w_inf, torch.ones_like(w_inf), atol=1e-3), \
        "γ→∞ 에서 표준 손실과 같아야 한다"
    w5 = min_snr_weight(diff.alphas_bar, tt, 5.0).flatten()
    assert (w5 <= 1.0 + 1e-6).all(), "가중치가 1 을 넘는다"
    assert w5[0] < w5[-1], "낮은 노이즈 구간이 낮게 가중되어야 한다"

    # ---- 7. EMA ----------------------------------------------------------
    m = ResUNet(channels=(4, 8))
    ema = EMA(m, decay=0.9)
    old = m.state_dict()["final_conv.weight"].clone()
    with torch.no_grad():                # 가중치를 바꿔야 평균이 따라가는지 알 수 있다
        for p_ in m.parameters():
            p_.add_(1.0)
    for _ in range(400):
        ema.update(m)
    new = m.state_dict()["final_conv.weight"]
    assert not torch.allclose(old, new), "점검 자체가 무의미하다 - 가중치가 안 바뀌었다"
    assert torch.allclose(ema.shadow["final_conv.weight"], new.float(), atol=1e-5), \
        "EMA 가 새 가중치로 수렴하지 않는다"
    ema.copy_to(m)                       # 평균을 모델에 다시 싣는 경로도 확인한다
    assert torch.allclose(m.state_dict()["final_conv.weight"], new, atol=1e-5), \
        "copy_to 가 가중치를 옮기지 못했다"

    # ---- 8. 학습률 워밍업 -------------------------------------------------
    assert lr_at(0, 2e-4, 0) == 2e-4, "워밍업 0 이면 기본 학습률이어야 한다"
    assert lr_at(0, 2e-4, 2000) < lr_at(1000, 2e-4, 2000) < lr_at(5000, 2e-4, 2000)
    assert abs(lr_at(5000, 2e-4, 2000) - 2e-4) < 1e-12, "워밍업 후 기본값에 도달해야 한다"

    print("자체 점검 8개 항목 통과")


self_check()

## 5. 데이터

본편이 만들어 둔 `data/cache/*.pt` 를 그대로 읽는다. 캐시가 없으면
먼저 본편 노트북의 4절을 한 번 실행해야 한다 (CelebA 디코딩에 시간이 걸린다).

In [ ]:
def load_cached(dataset, split):
    cfg = DATASETS[dataset]
    tag = cfg["limit"] if (cfg["limit"] and split == "train") else "all"
    path = CACHE_DIR / f"{dataset}_{split}_{cfg['image_size']}_{tag}.pt"
    if not path.exists():
        raise FileNotFoundError(
            f"{path} 가 없다. 본편 imagegeneration.ipynb 의 4절을 먼저 실행할 것")
    return torch.load(path, weights_only=True)


def make_loader(images, shuffle=True, batch_size=BATCH_SIZE):
    return DataLoader(TensorDataset(images), batch_size=batch_size, shuffle=shuffle,
                      drop_last=shuffle, pin_memory=(DEVICE.type == "cuda"))


def to_model_input(batch_uint8, augment=False):
    """uint8 (B,3,H,W) -> [-1,1] float. 증강은 GPU 에서 배치 단위로 한다."""
    x = batch_uint8.to(DEVICE, non_blocking=True).float().div_(127.5).sub_(1.0)
    if augment:
        flip = torch.rand(x.size(0), device=x.device) < 0.5
        x = torch.where(flip[:, None, None, None], x.flip(-1), x)
    return x.contiguous(memory_format=torch.channels_last)


DATA = {}
for name in DATASETS:
    DATA[name] = {s: load_cached(name, s) for s in ("train", "test")}
    print(f"{name}: train {tuple(DATA[name]['train'].shape)} | "
          f"test {tuple(DATA[name]['test'].shape)}")

## 6. 설정 사다리

여섯 단계이며 **누적**이다. 각 단계는 앞 단계에 변경 하나를 더한다.
따라서 인접한 두 행의 차이가 그 변경 하나의 기여다.

| 단계 | 더해진 변경 | 에폭 | 폭 | 블록/단계 |
|---|---|---|---|---|
| **E0** | 없음 (본편 재현) | 30 | (64,128) | 1 |
| **E1** | 가중치 EMA | 30 | (64,128) | 1 |
| **E2** | 워밍업 · 클리핑 · 드롭아웃 | 30 | (64,128) | 1 |
| **E3** | min-SNR 가중 (γ=5) | 30 | (64,128) | 1 |
| **E4** | 학습 5배 | 150 | (64,128) | 1 |
| **E5** | 용량 확대 | 150 | (128,256,256) | 2 |

E0 는 본편 시드 0 을 그대로 재현하는 대조군이다. 본편의 FID 94.93 과
크게 다르면 파이프라인이 어긋난 것이므로, 이 값이 사다리 전체의 검증점이 된다.

E5 의 CIFAR-10 설정은 해상도 단계를 2 에서 3 으로 늘린다. 이는 본편 Conclusions 가
남긴 깊이 가설의 검증에도 해당한다. 다만 폭과 단계를 동시에 바꾸므로
이 사다리만으로 둘을 분리하지는 못한다.

### 예산 조절

`PRESET` 으로 규모를 고른다. 먼저 `"quick"` 으로 파이프라인이 도는지 확인하고
`"full"` 로 본 실험을 돌리기를 권한다.

| PRESET | 무엇을 도는가 | FID 표본 | RTX 4070 기준 |
|---|---|---|---|
| `"quick"` | CIFAR-10 만, 전 단계 10 에폭, E5 도 좁은 폭 | 512 | 약 40분 |
| `"full"` | 아래 계획대로 | 2048 | 약 13시간 |

`"full"` 의 계획은 이렇다.

* **CIFAR-10 — 사다리 전체.** 각 변경의 기여를 분리하는 일은 여기서 한다.
  E0~E3 는 30 에폭, E4 는 150 에폭, E5 는 120 에폭
* **CelebA — 양 끝만 (E0, E5).** 사다리를 두 번 돌릴 필요는 없다.
  결론이 다른 데이터셋으로 옮겨 가는지만 확인하면 되고, 그것은 시작점과
  끝점의 차이로 충분하다

에폭 단위로 이어서 학습하므로 한 번에 끝낼 필요는 없다. 중단했다가 같은 셀을
다시 실행하면 남은 것만 돈다.

### 표본 생성 비용 (실측, 조용한 GPU 로 환산)

FID 용 2048장을 조상 샘플러 1000 스텝으로 만드는 데 걸리는 시간이다.

| 모델 | 32×32 | 64×64 |
|---|---|---|
| 2.5M (E0~E4) | 약 4분 | 약 16분 |
| 31.5M / 24.4M (E5) | 약 21분 | 약 80분 |

전부 합쳐 `"full"` 13시간 중 약 18%다. `EVAL_SAMPLER = "ddim"` 으로 바꾸면
이것이 10분의 1로 줄지만(약 2시간 절약) 본편의 FID 94.93 과 비교할 수 없게
된다. 본편과의 연결이 사다리 전체의 검증점이므로 기본값은 조상 샘플러다.

In [ ]:
PRESET = "quick"        # "quick" 으로 먼저 확인한 뒤 "full" 로 바꿀 것

# 각 단계는 앞 단계를 상속하고 하나만 바꾼다. 아래 코드가 그 누적을 만든다
_STEPS = [
    ("E0", "기준선 재현",              dict()),
    ("E1", "+ EMA",                    dict(ema=0.9995)),
    ("E2", "+ 워밍업·클리핑·드롭아웃",  dict(warmup=2000, grad_clip=1.0, dropout=0.1)),
    ("E3", "+ min-SNR (γ=5)",          dict(min_snr=5.0)),
    ("E4", "+ 학습 5배",               dict(epochs=150)),
    ("E5", "+ 용량 확대",              dict(wide=True, blocks=2)),
]

_BASE = dict(epochs=30, ema=None, warmup=0, grad_clip=None, dropout=0.0,
             min_snr=None, wide=False, blocks=1)

# 넓힌 설정. CIFAR-10 은 단계를 2 -> 3 으로 늘린다
WIDE_CHANNELS = {
    "CIFAR-10": (128, 256, 256),
    "CelebA":   (96, 192, 256),
}

LADDER = []
_acc = dict(_BASE)
for _tag, _label, _delta in _STEPS:
    _acc = dict(_acc, **_delta)
    LADDER.append(dict(_acc, tag=_tag, label=_label))

if PRESET == "quick":
    # 파이프라인 점검용. 순위를 볼 수 있을 만큼만 돌린다
    DATASETS_TO_RUN = ["CIFAR-10"]
    TAGS_FOR = {"CIFAR-10": [c["tag"] for c in LADDER]}
    FID_SAMPLES = 512
    WIDE_CHANNELS = {"CIFAR-10": (64, 128, 128), "CelebA": (48, 96, 128)}
    for _c in LADDER:
        _c["epochs"] = 10
elif PRESET == "full":
    # CIFAR-10 은 사다리 전체, CelebA 는 양 끝만. 이유는 위 마크다운 참조
    DATASETS_TO_RUN = ["CIFAR-10", "CelebA"]
    TAGS_FOR = {"CIFAR-10": ["E0", "E1", "E2", "E3", "E4", "E5"],
                "CelebA":   ["E0", "E5"]}
    FID_SAMPLES = 2048
    LADDER[5]["epochs"] = 120           # 31M 모델은 150 에폭이면 4시간을 넘긴다
else:
    raise ValueError(PRESET)


def ladder_for(dataset):
    """이 데이터셋에서 실제로 돌릴 단계만 순서대로 준다."""
    return [c for c in LADDER if c["tag"] in TAGS_FOR[dataset]]


def channels_for(cfg, dataset):
    return WIDE_CHANNELS[dataset] if cfg["wide"] else DATASETS[dataset]["channels"]


def build_model(cfg, dataset):
    return ResUNet(channels=channels_for(cfg, dataset), dropout=cfg["dropout"],
                   blocks_per_stage=cfg["blocks"])


print(f"PRESET = {PRESET} | FID 표본 {FID_SAMPLES}")
print()
for _ds in DATASETS_TO_RUN:
    _rows = ladder_for(_ds)
    print(f"[{_ds}]  {len(_rows)}개 설정")
    print(f"  {pad('단계', 5)}{pad('변경', 26)}{'에폭':>6} {'폭':<18} {'블록':>5} {'파라미터':>12}")
    print("  " + "-" * 76)
    for _c in _rows:
        _n = count_params(build_model(_c, _ds))
        print(f"  {pad(_c['tag'], 5)}{pad(_c['label'], 26)}{_c['epochs']:>6} "
              f"{str(channels_for(_c, _ds)):<18} {_c['blocks']:>5} {_n:>12,}")
    print()

## 7. 학습

에폭이 끝날 때마다 모델·EMA·옵티마이저·전역 스텝을 저장한다.
중단 후 같은 셀을 다시 실행하면 **남은 에폭만** 이어서 돈다.

동시 실행 잠금을 둔다. 본편에서 두 곳이 같은 체크포인트에 동시에 쓰는 사고가
있었고, 그때 나온 수치는 폐기해야 했다. 잠금 파일이 남아 실행이 막히면
`(EXT_DIR / "train.lock").unlink()` 로 지운다.

### 속도에 대해 실제로 측정한 것

RTX 4070, 배치 128, CIFAR-10 32×32 에서 스텝 시간을 직접 쟀다. 아래는 추측이
아니라 측정값이다. 다만 GPU 를 데스크톱 응용과 공유한 상태여서 **절대값은 약
2.5배 부풀려져 있다**(같은 조건의 기준 설정이 본편 기록 16.0초 대신 40초로
나온다). 상대 비교만 의미가 있다.

**켜져 있는 것**

| 항목 | 측정된 효과 |
|---|---|
| bf16 혼합정밀도 | 157.4 → 107.7 ms/step (**1.46배**) |
| EMA 를 `_foreach` 로 묶기 | 2.5M 모델 **5.9%**, 31.5M 모델 **3.0%** |
| fused Adam | 약 1% |
| TF32 · cudnn autotune · uint8 캐시 · GPU 측 증강 | 본편에서 이미 적용 |

EMA 항목은 원래 텐서마다 `mul_`/`add_` 를 부르고 스텝마다 `state_dict()` 로 새
사전을 만들었다. 스텝의 6%를 잡아먹고 있었고, `_foreach` 로 묶으니 EMA 를 아예
끈 것과 같은 속도가 됐다(106.74 대 106.66 ms).

**측정해 보고 기각한 것**

| 항목 | 결과 | 이유 |
|---|---|---|
| 데이터셋 전체를 GPU 에 상주 | **−0.2% / +0.1%** | 호스트→디바이스 복사는 이미 병목이 아니다 |
| 스텝마다 `loss.item()` 제거 | 약 1% | 복잡도에 비해 이득이 없다 |
| 샘플링 배치 확대 (64→1024) | **0%** | 배치 64 에서 이미 GPU 가 포화한다 (0.287 → 0.303 ms/img) |
| `channels_last` | 차이 없음 | 이 구조에서는 도움이 되지 않는다. 본편과 맞추려고 유지 |

**남은 레버 하나**

`torch.compile` 은 이 환경에서 **Triton 이 없어 실패한다**(`TritonMissing`).
Windows 용 빌드를 설치하면 쓸 수 있고, 이 정도 컨볼루션 신경망에서 보통 10~30%
를 준다.

```
pip install triton-windows
```

설치했다면 설정 셀의 `USE_COMPILE = True` 로 바꾸면 된다. 실패해도 원본 모델로
그대로 진행하므로 학습이 멈추지는 않는다. **이 환경에서 검증하지 못한 유일한
항목**이라는 점은 감안할 것.

### 정직하게 말하면

공학적 최적화로 남은 여지는 크지 않다. 전체 시간을 지배하는 것은 **E4 의 150
에폭과 E5 의 31.5M 파라미터**이며, 이것들은 줄이려고 넣은 것이 아니라 원인 1
(학습 예산 부족)을 직접 겨냥해 넣은 것이다. 시간을 줄이고 싶다면 손댈 곳은
커널이 아니라 이 두 숫자다.

In [ ]:
LOCK = EXT_DIR / "train.lock"
LOCK_STALE_HOURS = 12


def acquire_lock():
    if LOCK.exists():
        age = (time.time() - LOCK.stat().st_mtime) / 3600
        if age < LOCK_STALE_HOURS:
            raise RuntimeError(
                f"{LOCK} 가 {age:.1f}시간 전부터 있다. 다른 곳에서 학습 중일 수 있다. "
                f"아니라면 LOCK.unlink() 로 지우고 다시 실행할 것")
        print(f"오래된 잠금({age:.1f}시간)을 무시한다")
    LOCK.write_text(f"pid {os.getpid()} @ {time.ctime()}", encoding="utf-8")


def release_lock():
    if LOCK.exists():
        LOCK.unlink()


def ext_ckpt(dataset, tag):
    d = EXT_DIR / dataset
    d.mkdir(parents=True, exist_ok=True)
    return d / f"{tag}.pt"


def train_config(cfg, dataset, loader):
    """한 설정을 학습한다. 이미 끝난 것은 건너뛰고, 도중까지 된 것은 이어서 한다."""
    ckpt = ext_ckpt(dataset, cfg["tag"])
    model = build_model(cfg, dataset).to(DEVICE, memory_format=torch.channels_last)
    if USE_COMPILE:
        try:
            model = torch.compile(model)
        except Exception as exc:            # Triton 미설치 등. 그냥 원본으로 간다
            print("  torch.compile 사용 불가:", exc)
    diffusion = GaussianDiffusion(model, num_steps=NUM_STEPS).to(DEVICE)
    # fused Adam 은 갱신을 커널 하나로 묶는다. 지원하지 않는 조합이면 물러선다
    try:
        opt = torch.optim.Adam(model.parameters(), lr=BASE_LR, fused=True)
    except (RuntimeError, ValueError):
        opt = torch.optim.Adam(model.parameters(), lr=BASE_LR)
    ema = EMA(model, cfg["ema"]) if cfg["ema"] else None

    start_epoch, gstep, history = 0, 0, []
    if ckpt.exists():
        state = torch.load(ckpt, map_location=DEVICE, weights_only=False)
        if state["epoch"] >= cfg["epochs"]:
            print(f"  [{dataset}/{cfg['tag']}] 이미 {state['epoch']} 에폭 완료 - 건너뜀 "
                  f"(loss {state['history'][-1]:.5f})")
            return state["history"]
        raw(model).load_state_dict(state["model"])
        opt.load_state_dict(state["opt"])
        if ema and state.get("ema"):
            ema.load_state_dict(state["ema"], model)   # 목록을 다시 묶어야 한다
        start_epoch, gstep, history = state["epoch"], state["gstep"], state["history"]
        print(f"  [{dataset}/{cfg['tag']}] {start_epoch} 에폭에서 이어서 학습")

    for epoch in range(start_epoch + 1, cfg["epochs"] + 1):
        model.train()
        running, seen, t0 = 0.0, 0, time.time()

        for (batch,) in loader:
            data = to_model_input(batch, augment=True)
            for g in opt.param_groups:
                g["lr"] = lr_at(gstep, BASE_LR, cfg["warmup"])
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEVICE.type, dtype=AMP_DTYPE, enabled=USE_AMP):
                loss = diffusion.p_losses(data, cfg["min_snr"])
            loss.backward()
            if cfg["grad_clip"]:
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg["grad_clip"])
            opt.step()
            gstep += 1
            if ema:
                ema.update()

            running += loss.item() * data.size(0)
            seen += data.size(0)

        history.append(running / seen)
        dt = time.time() - t0
        torch.save({"model": raw(model).state_dict(), "opt": opt.state_dict(),
                    "ema": ema.state_dict() if ema else None,
                    "epoch": epoch, "gstep": gstep, "history": history,
                    "cfg": {k: v for k, v in cfg.items()}, "dataset": dataset}, ckpt)

        if epoch == start_epoch + 1:
            left = (cfg["epochs"] - epoch) * dt / 60
            print(f"  epoch {epoch:3d} | loss {history[-1]:.5f} | {dt:.1f}s "
                  f"| 남은 예상 {left:.0f}분")
        elif epoch % 5 == 0 or epoch == cfg["epochs"]:
            print(f"  epoch {epoch:3d} | loss {history[-1]:.5f} | {dt:.1f}s")

    return history


acquire_lock()
try:
    HISTORIES = {}
    for dataset in DATASETS_TO_RUN:
        loader = make_loader(DATA[dataset]["train"])
        for cfg in ladder_for(dataset):
            print(f"[{dataset} / {cfg['tag']} {cfg['label']}]")
            torch.manual_seed(SEED)     # 모든 설정이 같은 초기 조건에서 출발한다
            HISTORIES[(dataset, cfg["tag"])] = train_config(cfg, dataset, loader)
finally:
    release_lock()

print("\n학습 완료")

## 8. 평가

각 설정에 대해 다음을 잰다.

* **FID** — 본편과 같은 조상 샘플러(1000 스텝), 같은 표본 수. 낮을수록 좋다
* **Inception Score** — 높을수록 좋다
* **DDIM 대조** — 최종 설정에 한해 DDIM 100 스텝으로도 재서, 10배 빠른 샘플링이
  품질을 얼마나 잃는지 본다

EMA 를 쓴 설정은 **평균 가중치로** 표본을 만든다. EMA 의 의미가 거기에 있다.

In [ ]:
from torchmetrics.image.fid import FrechetInceptionDistance
from torchmetrics.image.inception import InceptionScore


def load_for_eval(cfg, dataset):
    """체크포인트를 읽어 확산 객체를 만든다. EMA 가 있으면 평균 가중치를 싣는다."""
    state = torch.load(ext_ckpt(dataset, cfg["tag"]), map_location=DEVICE,
                       weights_only=False)
    model = build_model(cfg, dataset).to(DEVICE, memory_format=torch.channels_last)
    model.load_state_dict(state["model"])
    if cfg["ema"] and state.get("ema"):
        ema = EMA(model, cfg["ema"])
        ema.load_state_dict(state["ema"], model)
        ema.copy_to(model)
    model.eval()
    return GaussianDiffusion(model, num_steps=NUM_STEPS).to(DEVICE), state


@torch.no_grad()
def generate(diffusion, n, image_size, sampler="ancestral", ddim_steps=100,
             batch=128, seed=1234):
    """표본을 uint8 로 모아 돌려준다. 시드를 고정해 설정 간 시작 노이즈를 맞춘다."""
    g = torch.Generator(device=DEVICE).manual_seed(seed)
    out = torch.empty(n, 3, image_size, image_size, dtype=torch.uint8)
    done = 0
    while done < n:
        b = min(batch, n - done)
        shape = (b, 3, image_size, image_size)
        if sampler == "ancestral":
            x = diffusion.p_sample_loop(shape, device=DEVICE, generator=g)
        else:
            x = diffusion.ddim_sample(shape, steps=ddim_steps, eta=0.0,
                                      device=DEVICE, generator=g)
        out[done:done + b] = ((x.clamp(-1, 1) + 1) * 127.5).round().to(torch.uint8).cpu()
        done += b
    return out


@torch.no_grad()
def fid_is(fake_uint8, real_uint8, batch=64):
    """torchmetrics 로 FID 와 IS 를 잰다. 둘 다 uint8 [0,255] 를 요구한다."""
    fid = FrechetInceptionDistance(feature=2048, normalize=False).to(DEVICE)
    isc = InceptionScore(normalize=False).to(DEVICE)
    real = real_uint8[:len(fake_uint8)]
    for i in range(0, len(real), batch):
        fid.update(real[i:i + batch].to(DEVICE), real=True)
    for i in range(0, len(fake_uint8), batch):
        chunk = fake_uint8[i:i + batch].to(DEVICE)
        fid.update(chunk, real=False)
        isc.update(chunk)
    m, s = isc.compute()
    return fid.compute().item(), m.item(), s.item()


RESULTS = json.loads(RESULT_JSON.read_text(encoding="utf-8")) if RESULT_JSON.exists() else {}

for dataset in DATASETS_TO_RUN:
    size = DATASETS[dataset]["image_size"]
    real = DATA[dataset]["test"]
    for cfg in ladder_for(dataset):
        key = f"{dataset}|{cfg['tag']}|{EVAL_SAMPLER}"
        if key in RESULTS:
            print(f"{key}: 이미 계산됨 (FID {RESULTS[key]['fid']:.2f})")
            continue
        t0 = time.time()
        diffusion, state = load_for_eval(cfg, dataset)
        fake = generate(diffusion, FID_SAMPLES, size, sampler=EVAL_SAMPLER,
                        ddim_steps=EVAL_DDIM_STEPS)
        f, im, isd = fid_is(fake, real)
        RESULTS[key] = dict(fid=f, is_mean=im, is_std=isd,
                            loss_final=state["history"][-1], epochs=state["epoch"],
                            params=count_params(diffusion.model),
                            minutes=(time.time() - t0) / 60)
        RESULT_JSON.write_text(json.dumps(RESULTS, indent=2, ensure_ascii=False),
                               encoding="utf-8")
        print(f"{key}: FID {f:7.2f} | IS {im:.3f} | {(time.time() - t0) / 60:.1f}분")

# ---- 최종 설정만 DDIM 으로 한 번 더 --------------------------------------
for dataset in DATASETS_TO_RUN:
    size = DATASETS[dataset]["image_size"]
    cfg = ladder_for(dataset)[-1]
    key = f"{dataset}|{cfg['tag']}|ddim100"
    if key in RESULTS:
        continue
    t0 = time.time()
    diffusion, state = load_for_eval(cfg, dataset)
    fake = generate(diffusion, FID_SAMPLES, size, sampler="ddim", ddim_steps=100)
    f, im, isd = fid_is(fake, DATA[dataset]["test"])
    RESULTS[key] = dict(fid=f, is_mean=im, is_std=isd, epochs=state["epoch"],
                        params=count_params(diffusion.model),
                        minutes=(time.time() - t0) / 60)
    RESULT_JSON.write_text(json.dumps(RESULTS, indent=2, ensure_ascii=False),
                           encoding="utf-8")
    print(f"{key}: FID {f:7.2f} | IS {im:.3f} | {(time.time() - t0) / 60:.1f}분 "
          f"(조상 샘플러 대비 스텝 1/10)")

## 9. 결과 표

각 행은 앞 행에 변경 하나를 더한 것이므로, **ΔFID 열이 그 변경 하나의 기여**다.
본편 시드 0 의 ResUNet FID 를 기준선으로 함께 적는다.

In [ ]:
def results_table(dataset):
    base = BASELINE.get(dataset, {}).get("fid")
    rows, prev = [], None
    for cfg in ladder_for(dataset):
        r = RESULTS.get(f"{dataset}|{cfg['tag']}|{EVAL_SAMPLER}")
        if not r:
            continue
        delta = "" if prev is None else f"{r['fid'] - prev:+7.2f}"
        rows.append((cfg["tag"], cfg["label"], r["epochs"], r["params"],
                     r["loss_final"], r["fid"], delta, r["is_mean"]))
        prev = r["fid"]

    print(f"=== {dataset} ===")
    if base:
        print(f"본편 시드 0 ResUNet: FID {base:.2f}")
    print()
    print(f"{pad('단계', 5)}{pad('변경', 26)}{'에폭':>6} {'파라미터':>12} {'손실':>9} "
          f"{'FID':>8} {'ΔFID':>8} {'IS':>7}")
    print("-" * 88)
    for tag, label, ep, pr, ls, fd, dl, ism in rows:
        print(f"{pad(tag, 5)}{pad(label, 26)}{ep:>6} {pr:>12,} {ls:>9.5f} "
              f"{fd:>8.2f} {dl:>8} {ism:>7.3f}")

    ddim = RESULTS.get(f"{dataset}|{ladder_for(dataset)[-1]['tag']}|ddim100")
    if ddim and rows:
        print("-" * 88)
        print(f"{pad('', 5)}{pad('최종 설정 + DDIM 100 스텝', 26)}{ddim['epochs']:>6} "
              f"{ddim['params']:>12,} {'':>9} {ddim['fid']:>8.2f} "
              f"{ddim['fid'] - rows[-1][5]:>+8.2f} {ddim['is_mean']:>7.3f}")
    if base and rows:
        print()
        print(f"총 개선: FID {base:.2f} -> {rows[-1][5]:.2f} "
              f"({(1 - rows[-1][5] / base) * 100:.1f}% 감소)")
    print()


for dataset in DATASETS_TO_RUN:
    results_table(dataset)

## 10. 표본 비교 그림

논문에 넣을 그림을 만든다. 모든 설정이 **같은 시작 노이즈**를 쓰므로
칸 사이의 차이는 학습 설정에서만 온다.

In [ ]:
def sample_grid_figure(dataset, tags=None, n=64, seed=7):
    """설정별 표본 격자를 가로로 늘어놓아 하나의 그림으로 저장한다."""
    cfgs = [c for c in ladder_for(dataset) if tags is None or c["tag"] in tags]
    size = DATASETS[dataset]["image_size"]

    fig, axes = plt.subplots(1, len(cfgs), figsize=(4 * len(cfgs), 4.6))
    axes = np.atleast_1d(axes)
    for ax, cfg in zip(axes, cfgs):
        if not ext_ckpt(dataset, cfg["tag"]).exists():
            ax.axis("off")
            continue
        diffusion, _ = load_for_eval(cfg, dataset)
        imgs = generate(diffusion, n, size, sampler=EVAL_SAMPLER,
                        ddim_steps=EVAL_DDIM_STEPS, seed=seed)
        grid = make_grid(imgs.float() / 255.0, nrow=8)
        ax.imshow(grid.permute(1, 2, 0).numpy())
        r = RESULTS.get(f"{dataset}|{cfg['tag']}|{EVAL_SAMPLER}")
        ax.set_title(f"{cfg['tag']}  {cfg['label']}\n"
                     + (f"FID {r['fid']:.1f}" if r else ""), fontsize=11)
        ax.axis("off")

    fig.suptitle(f"{dataset}: samples by training configuration "
                 "(identical starting noise)", fontsize=13)
    fig.tight_layout()
    out = FIG_DIR / f"ext_samples_{dataset.lower().replace('-', '')}.png"
    fig.savefig(out, dpi=140, bbox_inches="tight")
    plt.show()
    print("저장:", out)


def fid_ladder_figure():
    """단계별 FID 변화를 막대로 그린다."""
    fig, axes = plt.subplots(1, len(DATASETS_TO_RUN),
                             figsize=(6 * len(DATASETS_TO_RUN), 4), squeeze=False)
    for ax, dataset in zip(axes[0], DATASETS_TO_RUN):
        tags, vals = [], []
        for cfg in ladder_for(dataset):
            r = RESULTS.get(f"{dataset}|{cfg['tag']}|{EVAL_SAMPLER}")
            if r:
                tags.append(cfg["tag"])
                vals.append(r["fid"])
        ax.bar(tags, vals, color="#4C72B0")
        if BASELINE.get(dataset):
            ax.axhline(BASELINE[dataset]["fid"], ls="--", c="crimson",
                       label=f"Chapter 3 baseline {BASELINE[dataset]['fid']:.1f}")
            ax.legend()
        for i, v in enumerate(vals):
            ax.text(i, v, f"{v:.1f}", ha="center", va="bottom", fontsize=9)
        ax.set_title(dataset)
        ax.set_ylabel("FID (lower is better)")
    fig.tight_layout()
    out = FIG_DIR / "ext_fid_ladder.png"
    fig.savefig(out, dpi=140, bbox_inches="tight")
    plt.show()
    print("저장:", out)


for dataset in DATASETS_TO_RUN:
    sample_grid_figure(dataset)
fid_ladder_figure()

## 11. 논문에 무엇을 쓸 것인가

이 노트북이 끝나면 다음이 남는다.

* `model/extension/extension_study.json` — 모든 수치. 논문 표의 유일한 출처
* `dissertation/imgs/ext_samples_*.png` — 설정별 표본 격자
* `dissertation/imgs/ext_fid_ladder.png` — 단계별 FID 막대

### 쓸 때 지켜야 할 것

1. **ΔFID 열만이 인과적 주장을 지지한다.** 사다리가 누적이므로 인접 행의 차이는
   그 변경 하나에 귀속된다. 반면 행의 절대값은 앞선 모든 변경을 포함한다.
2. **시드는 하나다.** 본편이 보였듯 FID 의 시드 간 표준편차는 7.8~32.9 다.
   ΔFID 가 그보다 작은 단계는 **효과가 있다고 말할 수 없다**. 표에 그대로 적고,
   유의성을 주장하지 말 것.
3. **E0 가 본편의 94.93 과 크게 다르면** 다른 모든 행을 신뢰할 수 없다. 먼저 그것부터 확인한다.
4. **E5 는 폭과 해상도 단계를 동시에 바꾼다.** 깊이 가설의 검증으로 쓸 수 없다.
   그 실험은 폭을 고정한 채 단계만 2 에서 3 으로 바꿔야 한다.

### 아직 하지 않은 것

* **v 예측 / x0 예측** — 목적함수 자체를 바꾸면 평균 회귀의 성질이 달라진다.
  이 노트북은 ε 예측을 유지했다
* **분류자 없는 안내(CFG)** — 무조건 생성이라 적용 대상이 아니다. 조건부로 바꾸면
  선명도에 가장 큰 폭의 개선을 주는 항목이다
* **깊이 가설의 분리 실험** — 위 4번